<a href="https://colab.research.google.com/github/jasminealyaa1212/DIP-final-project/blob/main/coba%20dulu%20deh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import urllib.request
from collections import deque

import cv2
import numpy as np

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
except ImportError:
    pass  # running outside Colab

BASE = '/content/drive/MyDrive/semester 3/Digital Image Processing/final project'
VIDEO_PATH = f'{BASE}/video 2.mp4'
OUTPUT_PATH = f'{BASE}/annotated_output.mp4'
SNAP_DIR = f'{BASE}/violations'

# ---- Direction of the lane -------------------------------------------------
# 'up'   = correct traffic moves toward the TOP of the frame (away from camera)
# 'down' = correct traffic moves toward the BOTTOM of the frame (toward camera)
# NOTE: in image coordinates y grows DOWNWARD, so "up" means dy < 0.
ALLOWED_DIRECTION = 'up'

# ---- Processing ------------------------------------------------------------
TARGET_W, TARGET_H = 1280, 720
FRAME_SKIP = 2               # process every N-th frame
WARMUP_FRAMES = 25           # let background model settle before judging

# ---- Detection -------------------------------------------------------------
MOTION_OVERLAP = 0.20        # min fraction of a Haar box that must be "moving"
MIN_BLOB_AREA = 2500         # px^2, smallest moving blob accepted as a vehicle
USE_BLOB_FALLBACK = True     # use moving blobs the cascades missed
NMS_IOU = 0.35

# ---- Tracking & direction decision ----------------------------------------
HIST_LEN = 12                # centroid history kept per track
MIN_HIST = 6                 # samples needed before judging
MAX_MISSED = 6               # frames a track may vanish before deletion
MIN_TRAVEL_RATIO = 0.35      # net vertical travel must exceed this * box height
VERTICAL_BIAS = 0.6          # |dy| must exceed this * |dx| (ignore sideways motion)
VOTES_TO_FLAG = 3            # consecutive wrong-way judgements -> violation

CASCADE_URLS = {
    'car': 'https://raw.githubusercontent.com/andrewssobral/vehicle_detection_haarcascades/master/cars.xml',
    'two_wheeler': 'https://raw.githubusercontent.com/AdityaPai2398/Vehicle-And-Pedestrian-Detection-Using-Haar-Cascades/master/Main%20Project/Main%20Project/Bike%20Detection/two_wheeler.xml',
    'bus': 'https://raw.githubusercontent.com/AdityaPai2398/Vehicle-And-Pedestrian-Detection-Using-Haar-Cascades/master/Main%20Project/Main%20Project/Bus%20Detection/Bus_front.xml',
}


def load_cascades():
    classifiers = {}
    for label, url in CASCADE_URLS.items():
        fname = f'{label}.xml'
        try:
            if not os.path.exists(fname):
                urllib.request.urlretrieve(url, fname)
            c = cv2.CascadeClassifier(fname)
            if not c.empty():
                classifiers[label] = c
                print(f'Loaded cascade: {label}')
        except Exception as e:
            print(f'Could not load {label}: {e}')
    return classifiers


# %% CELL 2 - Helper functions (preprocessing, motion, NMS, tracker)

def preprocess(frame):
    """Gray -> CLAHE (contrast) -> Gaussian blur (noise)."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    return cv2.GaussianBlur(gray, (5, 5), 0)


def motion_mask(subtractor, frame):
    """MOG2 background subtraction + morphology -> clean binary motion mask."""
    fg = subtractor.apply(frame)
    _, fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)   # drop shadow pixels (127)
    k_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    k_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, k_small)        # remove speckle noise
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, k_big)         # fill holes in vehicles
    fg = cv2.dilate(fg, k_small, iterations=2)
    return fg


def iou(a, b):
    ax, ay, aw, ah = a[:4]
    bx, by, bw, bh = b[:4]
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0


def nms(dets, thr=NMS_IOU):
    """Merge overlapping boxes (several cascades often fire on one vehicle)."""
    dets = sorted(dets, key=lambda d: d[2] * d[3], reverse=True)
    keep = []
    for d in dets:
        if all(iou(d, k) < thr for k in keep):
            keep.append(d)
    return keep


def detect_vehicles(gray, fg_mask, classifiers):
    """Haar detections confirmed by motion, plus optional blob fallback."""
    raw = []
    for label, cascade in classifiers.items():
        min_sz = (20, 20) if label == 'two_wheeler' else (40, 40)
        for (x, y, w, h) in cascade.detectMultiScale(
                gray, scaleFactor=1.1, minNeighbors=4, minSize=min_sz):
            region = fg_mask[y:y + h, x:x + w]
            if region.size and (region > 0).mean() >= MOTION_OVERLAP:
                raw.append((x, y, w, h, label))      # moving AND looks like a vehicle

    if USE_BLOB_FALLBACK:
        contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in contours:
            if cv2.contourArea(c) >= MIN_BLOB_AREA:
                x, y, w, h = cv2.boundingRect(c)
                if all(iou((x, y, w, h), r) < 0.3 for r in raw):
                    raw.append((x, y, w, h, 'vehicle'))
    return nms(raw)


class Track:
    def __init__(self, tid, box, label):
        self.id = tid
        self.box = box
        self.label = label
        self.hist = deque(maxlen=HIST_LEN)
        self.missed = 0
        self.votes = 0
        self.flagged = False
        self.snapshot_saved = False
        self.add(box)

    @staticmethod
    def center(box):
        x, y, w, h = box
        return (x + w / 2.0, y + h / 2.0)

    def add(self, box):
        self.box = box
        self.hist.append(self.center(box))

    def judge(self):
        """Return 'wrong', 'ok' or None (not enough evidence)."""
        if len(self.hist) < MIN_HIST:
            return None
        (x0, y0), (x1, y1) = self.hist[0], self.hist[-1]
        dx, dy = x1 - x0, y1 - y0
        h = self.box[3]
        if abs(dy) < MIN_TRAVEL_RATIO * h:          # barely moved: stationary / noise
            return None
        if abs(dy) < VERTICAL_BIAS * abs(dx):       # mostly sideways: lane change/turn
            return None
        moving_up = dy < 0
        allowed_up = (ALLOWED_DIRECTION == 'up')
        return 'ok' if moving_up == allowed_up else 'wrong'


class CentroidTracker:
    def __init__(self):
        self.tracks = {}
        self.next_id = 0

    def update(self, dets):
        track_list = list(self.tracks.values())
        pairs = []
        for ti, t in enumerate(track_list):
            tc = t.center(t.box)
            for di, d in enumerate(dets):
                dc = Track.center(d[:4])
                dist = np.hypot(tc[0] - dc[0], tc[1] - dc[1])
                limit = max(60, 1.5 * d[3])         # allowed jump grows with vehicle size
                if dist < limit:
                    pairs.append((dist, ti, di))
        pairs.sort()

        used_t, used_d = set(), set()
        for dist, ti, di in pairs:                   # greedy nearest-neighbour matching
            if ti in used_t or di in used_d:
                continue
            t = track_list[ti]
            t.add(dets[di][:4])
            t.label = dets[di][4]
            t.missed = 0
            used_t.add(ti)
            used_d.add(di)

        for ti, t in enumerate(track_list):
            if ti not in used_t:
                t.missed += 1
        for di, d in enumerate(dets):
            if di not in used_d:
                self.tracks[self.next_id] = Track(self.next_id, d[:4], d[4])
                self.next_id += 1

        for tid in [k for k, t in self.tracks.items() if t.missed > MAX_MISSED]:
            del self.tracks[tid]
        return self.tracks


# %% CELL 3 - Main loop (streams frames straight to disk, no RAM blow-up)
def run():
    classifiers = load_cascades()
    if not os.path.exists(VIDEO_PATH):
        raise FileNotFoundError(f'Video not found: {VIDEO_PATH}')
    os.makedirs(SNAP_DIR, exist_ok=True)

    cap = cv2.VideoCapture(VIDEO_PATH)
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'),
                          max(1, src_fps / FRAME_SKIP), (TARGET_W, TARGET_H))

    subtractor = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=32,
                                                    detectShadows=True)
    tracker = CentroidTracker()
    violators, seen_ids = set(), set()
    montage_saved = False
    frame_idx = processed = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_idx += 1
        if frame_idx % FRAME_SKIP:
            continue

        frame = cv2.resize(frame, (TARGET_W, TARGET_H))
        gray = preprocess(frame)
        fg = motion_mask(subtractor, frame)
        processed += 1

        if processed <= WARMUP_FRAMES:               # background model still learning
            out.write(frame)
            continue

        dets = detect_vehicles(gray, fg, classifiers)
        tracks = tracker.update(dets)

        for t in tracks.values():
            if t.missed > 0:
                continue                             # not seen this frame
            seen_ids.add(t.id)
            verdict = t.judge()
            if verdict == 'wrong':
                t.votes += 1
            elif verdict == 'ok':
                t.votes = 0
            if t.votes >= VOTES_TO_FLAG:
                t.flagged = True                     # sticky: stays red once flagged
                violators.add(t.id)

            x, y, w, h = [int(v) for v in t.box]
            color = (0, 0, 255) if t.flagged else (0, 255, 0)
            tag = 'WRONG WAY' if t.flagged else 'NORMAL'
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, f'#{t.id} {tag} [{t.label}]', (x, max(15, y - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            pts = np.array(t.hist, dtype=np.int32).reshape(-1, 1, 2)
            cv2.polylines(frame, [pts], False, color, 2)   # trajectory trail

            if t.flagged and not t.snapshot_saved:
                cv2.imwrite(f'{SNAP_DIR}/violation_id{t.id}_f{frame_idx}.jpg', frame)
                t.snapshot_saved = True
                if not montage_saved:                # one figure for the report
                    mask_bgr = cv2.cvtColor(fg, cv2.COLOR_GRAY2BGR)
                    gray_bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
                    strip = np.hstack([cv2.resize(p, (640, 360))
                                       for p in (gray_bgr, mask_bgr, frame)])
                    cv2.imwrite(f'{SNAP_DIR}/pipeline_montage.jpg', strip)
                    montage_saved = True

        # HUD
        arrow = 'UP' if ALLOWED_DIRECTION == 'up' else 'DOWN'
        cv2.rectangle(frame, (0, 0), (430, 70), (0, 0, 0), -1)
        cv2.putText(frame, f'Allowed direction: {arrow}', (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f'Vehicles: {len(seen_ids)}   Wrong-way: {len(violators)}',
                    (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)
        out.write(frame)

    cap.release()
    out.release()
    print(f'Done. Processed {processed} frames.')
    print(f'Unique vehicles tracked: {len(seen_ids)} | wrong-way: {len(violators)}')
    print(f'Video: {OUTPUT_PATH}\nSnapshots: {SNAP_DIR}')


# %% CELL 4 - Run
run()


Loaded cascade: car
Loaded cascade: two_wheeler
Loaded cascade: bus
Done. Processed 190 frames.
Unique vehicles tracked: 25 | wrong-way: 3
Video: /content/drive/MyDrive/semester 3/Digital Image Processing/final project/annotated_output.mp4
Snapshots: /content/drive/MyDrive/semester 3/Digital Image Processing/final project/violations
